# Overfit & Data-Leakage Diagnostics
This notebook runs a set of automated checks to help determine whether your high accuracy is genuine or caused by leakage/duplicates/non-patient splits. It performs:
- filename overlap check between Training and Testing
- exact duplicate detection (SHA256)
- perceptual / near-duplicate detection (phash)
- patient-id overlap heuristics and GroupKFold split diagnostics
- saves a small CSV summary report in the repository root

Run cells in order. The notebook is intentionally read-only (no retraining).

In [1]:
# Imports and optional installers
import os
import sys
from pathlib import Path
import hashlib
import re
import json
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# optional: imagehash for perceptual hashing
try:
    import imagehash
    HAS_IMAGEHASH = True
except Exception:
    HAS_IMAGEHASH = False
    print("imagehash not available. To enable perceptual-duplicate checks run: pip install ImageHash Pillow")

# sklearn for GroupKFold
try:
    from sklearn.model_selection import GroupKFold
except Exception:
    GroupKFold = None
    print("scikit-learn not available. Install with: pip install scikit-learn")

# convenience
repo_root = Path('c:/Users/manav/Documents/GitHub/BrainTumorProject')  # adjust if needed
os.chdir(repo_root)
print('Working directory:', Path.cwd())

imagehash not available. To enable perceptual-duplicate checks run: pip install ImageHash Pillow
Working directory: c:\Users\manav\Documents\GitHub\BrainTumorProject


In [ ]:
# Configuration: processed dataset paths and classes (match your project layout)
# Use a raw string for Windows paths to avoid unicode-escape errors, or use forward slashes.
PROC_BASE = Path(r'C:\Users\manav\Documents\lighTumorNet\Brain_Tumor_MRI_Dataset\Processed_Brain_Tumor_MRI_Dataset')
TRAIN_DIR = PROC_BASE / 'Training'
TEST_DIR = PROC_BASE / 'Testing'
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Basic existence checks and counts
def count_images(root, classes=CLASS_NAMES):
    root = Path(root)
    counts = {}
    total = 0
    for c in classes:
        p = root / c
        if p.exists() and p.is_dir():
            n = len([f for f in p.iterdir() if f.suffix.lower() in ('.png','.jpg','.jpeg')])
        else:
            n = 0
        counts[c] = n
        total += n
    return counts, total

train_counts, train_total = count_images(TRAIN_DIR)
test_counts, test_total = count_images(TEST_DIR)

print('Training dir exists:', TRAIN_DIR.exists())
print('Testing dir exists :', TEST_DIR.exists())
print('Train total images:', train_total)
print('Test total images :', test_total)
print('\nTrain per-class counts:', train_counts)
print('Test per-class counts :', test_counts)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2124680525.py, line 2)

In [ ]:
# 1) Filename overlap check (train vs test)
def gather_filenames(root):
    root = Path(root)
    files = [p for p in root.rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg')]
    return files

train_files = gather_filenames(TRAIN_DIR)
test_files  = gather_filenames(TEST_DIR)
train_names = {p.name for p in train_files}
test_names  = {p.name for p in test_files}
overlap = train_names & test_names
print('Train files:', len(train_files))
print('Test files :', len(test_files))
print('Filename overlap count:', len(overlap))
if len(overlap)>0:
    print('Example overlaps:', list(overlap)[:10])

# Save small CSV of overlaps (if any)
if len(overlap)>0:
    pd.DataFrame({'overlap_filenames': list(overlap)[:100]}).to_csv('overlap_filenames_examples.csv', index=False)

# store results in a dict for final report
report = {}
report['train_count'] = len(train_files)
report['test_count']  = len(test_files)
report['filename_overlap_count'] = len(overlap)

Train files: 0
Test files : 0
Filename overlap count: 0


In [ ]:
# 2) Exact duplicate detection (SHA256). This can be slow; it hashes files binary-wise.
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

hash_map = defaultdict(list)
all_files = list(Path(PROC_BASE).rglob('*'))
img_files = [p for p in all_files if p.suffix.lower() in ('.png','.jpg','.jpeg')]
print('Hashing files (this may take a while):', len(img_files))
for i,p in enumerate(img_files):
    try:
        h = file_sha256(p)
        hash_map[h].append(str(p))
    except Exception as e:
        print('Failed hashing', p, e)

dupe_groups = {h:ps for h,ps in hash_map.items() if len(ps)>1}
print('Exact duplicate groups found:', len(dupe_groups))
# show up to 10 duplicate groups
for h,ps in list(dupe_groups.items())[:10]:
    print('--- group ---')
    for pp in ps:
        print(pp)

report['exact_duplicate_groups'] = len(dupe_groups)

Hashing files (this may take a while): 0
Exact duplicate groups found: 0


In [ ]:
# 3) Perceptual (near-duplicate) detection using phash (if imagehash is available)
near_dup_groups = {}
if HAS_IMAGEHASH:
    phash_map = defaultdict(list)
    for p in img_files:
        try:
            img = Image.open(p).convert('L')
            h = str(imagehash.phash(img))
            phash_map[h].append(str(p))
        except Exception as e:
            pass
    near_dup_groups = {h:ps for h,ps in phash_map.items() if len(ps)>1}
    print('Perceptual duplicate groups (phash) found:', len(near_dup_groups))
    for h,ps in list(near_dup_groups.items())[:10]:
        print('--- phash group ---')
        for pp in ps:
            print(pp)
else:
    print('Skipping perceptual duplicate detection because imagehash is not installed.')

report['near_duplicate_groups'] = len(near_dup_groups) if HAS_IMAGEHASH else None

Skipping perceptual duplicate detection because imagehash is not installed.


In [ ]:
# 4) Patient-ID overlap heuristics
# Try to extract patient IDs from filenames or parent folders. Adjust regex to your naming convention.
# Match formats like 'patient123', 'patient_123', 'pat-123', or 'patient(123)'
pid_re = re.compile(r'(?:patient|pat)[_-]?(?:\d+|\(\d+\))', re.I)

def extract_pid_from_path(p):
    # Try filename first
    m = pid_re.search(Path(p).name)
    if m:
        return m.group(0)
    # Then try parent directories (two levels up as heuristic)
    for part in Path(p).parts[-3:]:
        m2 = pid_re.search(part)
        if m2:
            return m2.group(0)
    return None

train_pids = set()
for p in train_files:
    pid = extract_pid_from_path(p)
    if pid:
        train_pids.add(pid)
test_pids = set()
for p in test_files:
    pid = extract_pid_from_path(p)
    if pid:
        test_pids.add(pid)

print('Extracted train patient ids (examples):', list(train_pids)[:10])
print('Extracted test patient ids  (examples):', list(test_pids)[:10])
overlap_pids = train_pids & test_pids
print('Patient id overlap count:', len(overlap_pids))
if overlap_pids:
    print('Example overlapping patient ids:', list(overlap_pids)[:10])

report['train_patient_ids_found'] = len(train_pids)
report['test_patient_ids_found'] = len(test_pids)
report['patient_id_overlap_count'] = len(overlap_pids)

error: unbalanced parenthesis at position 30

In [ ]:
# 5) GroupKFold diagnostics by patient (if enough patient ids were found)
if GroupKFold is None:
    print('sklearn GroupKFold not available; install scikit-learn to run grouping diagnostics')
else:
    # Build lists: paths, labels (from parent class folder), groups (patient ids or filename prefix)
    samples = []
    for c in CLASS_NAMES:
        for p in (TRAIN_DIR / c).glob('*'):
            if p.suffix.lower() in ('.png','.jpg','.jpeg'):
                samples.append((str(p), c))
    paths = [s[0] for s in samples]
    labels = [CLASS_NAMES.index(s[1]) for s in samples]
    groups = [extract_pid_from_path(p) or Path(p).stem.split('_')[0] for p in paths]
    print('Total samples used for grouping diagnostics:', len(paths))
    # Quick stats: how many unique groups?
    print('Unique groups found (example):', len(set(groups)))
    # Run GroupKFold splits and print val class distributions per fold
    gkf = GroupKFold(n_splits=5)
    for fold,(tr,va) in enumerate(gkf.split(paths, labels, groups)):
        val_labels = [labels[i] for i in va]
        dist = Counter(val_labels)
        dist_named = {CLASS_NAMES[k]: v for k,v in dist.items()}
        print(f'Fold {fold+1} val class dist: {dist_named}')

    # NOTE: to re-run a full GroupKFold CV of the model you would retrain per-fold using tr/va indexes.
    print('If GroupKFold distributions look reasonable, consider re-running CV using groups to ensure no patient leakage.')

In [ ]:
# 6) Save a compact report to CSV/JSON for quick review
summary = {
    'train_count': report.get('train_count'),
    'test_count': report.get('test_count'),
    'filename_overlap_count': report.get('filename_overlap_count'),
    'exact_duplicate_groups': report.get('exact_duplicate_groups'),
    'near_duplicate_groups': report.get('near_duplicate_groups'),
    'train_patient_ids_found': report.get('train_patient_ids_found'),
    'test_patient_ids_found': report.get('test_patient_ids_found'),
    'patient_id_overlap_count': report.get('patient_id_overlap_count')
}
pd.DataFrame([summary]).to_csv('overfit_diagnostics_summary.csv', index=False)
with open('overfit_diagnostics_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved summary to overfit_diagnostics_summary.csv and .json')
print(summary)

## Next steps / Interpretation
- If `filename_overlap_count`, `exact_duplicate_groups` or `patient_id_overlap_count` are non-zero: you likely have data leakage. Fix by re-splitting by patient and removing duplicates, then retrain.
- If no leakage is found and GroupKFold accuracy remains high, your model likely generalizes well on this dataset — still try an external holdout (different scanner/institution) if possible.
- If you want, I can extend this notebook to automatically re-run GroupKFold CV (retraining per-fold) or add perceptual duplicate thresholds and automatic de-duplication.